# Week 9, Day 1: Grids: lists of lists

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w009d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
board = []
for r in range(3):
    board.append(["."] * 3)
board[1][2] = "X"
for row in board:
    print("".join(row))

In [ ]:
seats = [
    ["A1", "A2", "A3", "A4"],
    ["B1", "B2", "B3", "B4"],
    ["C1", "C2", "C3", "C4"],
]
print(len(seats), "rows")
print(len(seats[0]), "columns")
print(seats[1])
print(seats[1][2])

*Interactive figure on the website: Each cell shows its value; the row and column numbers are along the edges. Try each view: a row is one inner list, but a column cuts across all of them..*

> **Key idea.** `grid[r]` is row `r` (an inner list). `grid[r][c]` is the item in row `r`, column `c`. The number of rows is `len(grid)`; the number of columns is `len(grid[0])`.

In [ ]:
def make_grid(rows, cols, fill=0):
    grid = []
    for _ in range(rows):
        grid.append([fill] * cols)
    return grid

g = make_grid(3, 4)
g[0][0] = 5
print(g)

In [ ]:
text = """1 2 3
4 5 6
7 8 9"""
grid = []
for line in text.splitlines():
    row = []
    for piece in line.split():
        row.append(int(piece))
    grid.append(row)
print(grid)

In [ ]:
grid = [[1, 2, 3], [4, 5, 6]]
total = 0
for row in grid:
    for value in row:
        total += value
print(total)

In [ ]:
grid = [[1, 2, 3], [4, 5, 6]]
for r in range(len(grid)):
    for c in range(len(grid[r])):
        grid[r][c] *= 2
print(grid)

In [ ]:
grid = [[1, 20, 3], [400, 5, 60]]
for row in grid:
    line = ""
    for value in row:
        line += f"{value:5}"
    print(line)

In [ ]:
grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
c = 1
column = []
for row in grid:
    column.append(row[c])
print("Row 0:", grid[0])
print("Column 1:", column)

In [ ]:
grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
n = len(grid)
main = []
other = []
for i in range(n):
    main.append(grid[i][i])
    other.append(grid[i][n - 1 - i])
print(main, other)

In [ ]:
grid = [[1, 2, 3], [4, 5, 6]]
flipped = []
for c in range(len(grid[0])):
    new_row = []
    for r in range(len(grid)):
        new_row.append(grid[r][c])
    flipped.append(new_row)
print(flipped)

In [ ]:
grid = [
    list("X.X"),
    list(".X."),
    list("XX."),
]

def count_neighbours(grid, r, c, mark="X"):
    count = 0
    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:
            if dr == 0 and dc == 0:
                continue
            nr, nc = r + dr, c + dc
            if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]):
                if grid[nr][nc] == mark:
                    count += 1
    return count

print(count_neighbours(grid, 1, 1))
print(count_neighbours(grid, 0, 0))

> **Negative indexes don't fail.** Without the bounds check, `grid[-1]` doesn't raise an error: it quietly gives the **last** row. So a cell on the top edge would "see" the bottom row as its neighbour. Always check `0 <= nr < rows` and `0 <= nc < cols` yourself.

### Exercise: Reach a cell

`plan` is a seating plan. Set `middle` to the item in the middle row and middle column, `corner` to the bottom-right item, and `back_row` to the whole last row, using indexes only.

In [ ]:
plan = [
    ["A1", "A2", "A3"],
    ["B1", "B2", "B3"],
    ["C1", "C2", "C3"],
]
middle = ___
corner = ___
back_row = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each value\"\n    assert middle == \"B2\" and corner == \"C3\" and back_row == [\"C1\", \"C2\", \"C3\"]\n    assert source.count('\"B2\"') == 1, \"Use indexes; don't type the values.\"")

### Exercise: Read a grid

Write `parse_grid(text)` that turns lines of whole numbers separated by spaces into a list of lists of ints.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several grids\"\n    assert parse_grid(\"1 2\\n3 4\") == [[1, 2], [3, 4]]\n    assert parse_grid(\"7\") == [[7]]\n    assert parse_grid(\"10 -2 3\\n0 0 0\") == [[10, -2, 3], [0, 0, 0]]")

### Exercise: Row and column totals

Write `row_totals(grid)` returning a list of the sum of each row, and `column_totals(grid)` returning a list of the sum of each column. All rows have the same length.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"row and column totals\"\n    g = [[1, 2, 3], [4, 5, 6]]\n    assert row_totals(g) == [6, 15]\n    assert column_totals(g) == [5, 7, 9]\n    assert column_totals([[4]]) == [4]")

### Exercise: Print a board

Write `show(board)` that **returns** the board as a string: each row's items joined by spaces, rows joined by `"\n"`. Items may be numbers or strings.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several boards\"\n    assert show([[\"X\", \".\", \"O\"], [\".\", \"X\", \".\"]]) == \"X . O\\n. X .\"\n    assert show([[1, 2], [3, 4]]) == \"1 2\\n3 4\"")

### Exercise: Transpose

Write `transpose(grid)` that returns a new grid with rows and columns swapped. A grid with 2 rows and 3 columns gives one with 3 rows and 2 columns.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several grids\"\n    assert transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]]\n    assert transpose([[1]]) == [[1]]\n    g = [[1, 2], [3, 4]]\n    t = transpose(g)\n    assert t == [[1, 3], [2, 4]] and g == [[1, 2], [3, 4]], \"Return a new grid.\"")

### Exercise: Three in a row

Write `winner(board)` for noughts and crosses. `board` is a 3 by 3 grid of `"X"`, `"O"` and `" "`. Return `"X"` or `"O"` if that player has three in a row (a row, a column or a diagonal), or `""` if nobody has.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several boards\"\n    assert winner([[\"X\", \"X\", \"X\"], [\" \", \"O\", \" \"], [\"O\", \" \", \" \"]]) == \"X\"\n    assert winner([[\"O\", \"X\", \" \"], [\"O\", \"X\", \" \"], [\"O\", \" \", \"X\"]]) == \"O\"\n    assert winner([[\"X\", \"O\", \" \"], [\" \", \"X\", \"O\"], [\" \", \" \", \"X\"]]) == \"X\"\n    assert winner([[\" \", \" \", \"O\"], [\" \", \"O\", \"X\"], [\"O\", \"X\", \" \"]]) == \"O\"\n    assert winner([[\"X\", \"O\", \"X\"], [\"X\", \"O\", \"O\"], [\"O\", \"X\", \"X\"]]) == \"\"\n    assert winner([[\" \"] * 3, [\" \"] * 3, [\" \"] * 3]) == \"\"")

### Exercise: Minesweeper numbers

In Minesweeper, each empty square shows how many mines touch it. Write `mine_counts(field)` where `field` is a list of strings, like `["*..", "..*", "..."]` (`*` is a mine). Return a new list of strings where every `.` is replaced by the number of mines among its 8 neighbours, and mines stay `*`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several fields\"\n    assert mine_counts([\"*..\", \"..*\", \"...\"]) == [\"*21\", \"12*\", \"011\"]\n    assert mine_counts([\"...\"]) == [\"000\"]\n    assert mine_counts([\"*\"]) == [\"*\"]\n    assert mine_counts([\"**\", \"**\"]) == [\"**\", \"**\"]")

### Bug hunt: Columns, not rows

`column(grid, c)` should return column `c`, but `column([[1, 2], [3, 4]], 0)` returns `[1, 2]`. Fix it.

In [ ]:
def column(grid, c):
    return grid[c]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several columns\"\n    assert column([[1, 2], [3, 4]], 0) == [1, 3]\n    assert column([[1, 2], [3, 4]], 1) == [2, 4]")

### Bug hunt: Seeing round the edge

`count_x` counts the `X`s next to a cell, but for the top-left cell of this board it counts the X in the bottom-right corner too. Fix the bounds check.

In [ ]:
def count_x(board, r, c):
    count = 0
    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:
            if dr == 0 and dc == 0:
                continue
            if board[r + dr][c + dc] == "X":
                count += 1
    return count

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"corners and middle\"\n    board = [list(\"...\"), list(\".X.\"), list(\"..X\")]\n    assert count_x(board, 0, 0) == 1, \"The top-left cell touches only the middle X.\"\n    assert count_x(board, 1, 1) == 1\n    assert count_x(board, 2, 2) == 1")

### Puzzle: Sum the main diagonal

Arrange the lines to add up the main diagonal of a square grid. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict the grid

Work these out without running them.

```python
g = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

a = g[2][0]
b = g[0][-1] + g[-1][0]
c = len(g[1])
d = g[1][1:]
```

In [ ]:
a = ___
b = ___
c = ___
d = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each value\"\n    assert a == 7 and b == 10 and c == 3 and d == [5, 6]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the Game of Life

In 1970 the mathematician John Conway invented the **Game of Life**: a grid of cells, each alive (`#`) or dead (`.`). Each generation, every cell looks at its 8 neighbours:

- a live cell with 2 or 3 live neighbours survives; otherwise it dies;
- a dead cell with exactly 3 live neighbours comes alive.

All cells change **at the same moment**, so you must build the new grid from the old one, never changing the old one as you go. Cells off the edge count as dead.

Write `step(grid)` that takes a list of strings and returns the next generation as a new list of strings.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_blinker():\n    \"a blinker flips between two shapes\"\n    a = [\".....\", \"..#..\", \"..#..\", \"..#..\", \".....\"]\n    b = [\".....\", \".....\", \".###.\", \".....\", \".....\"]\n    assert step(a) == b and step(b) == a\ndef test_block():\n    \"a block never changes\"\n    blk = [\"....\", \".##.\", \".##.\", \"....\"]\n    assert step(blk) == blk\ndef test_glider():\n    \"a glider moves\"\n    g = [\".#....\", \"..#...\", \"###...\", \"......\", \"......\"]\n    for _ in range(4):\n        g = step(g)\n    assert g == [\"......\", \"..#...\", \"...#..\", \".###..\", \"......\"], \"After four steps a glider moves one square down and one right.\"\ndef test_unchanged():\n    \"the input is not changed\"\n    a = [\"##\", \"##\"]\n    step(a)\n    assert a == [\"##\", \"##\"]")

## Warm-up

You built that safe grid last week, when you learned why `[["."] * 3] * 3` goes wrong. Today grids get a lesson of their own.

## Rows and cells

A grid is a list whose items are lists, one per **row**. Writing it out with one row per line shows its shape:

`seats[1]` is the whole second row, which is itself a list. `seats[1][2]` takes that row and then its item at position 2. Always **row first, then column**.

Explore the indexes. Click a cell, then choose what to highlight:

### Building a grid

You already know the safe way. Each row must be a **separate** list:

Grids often come from text: one line per row, with the items separated by spaces:

## Walking through a grid

To visit every cell, use nested loops: the outer one over the rows, the inner one over the items in a row:

When you need the positions (to change a cell, or to know where you are), loop over the indexes:

### Printing a grid neatly

Printing a grid directly shows brackets and commas. To print it as a table, build each row's line and print that:

## Rows, columns and diagonals

A **row** is easy, since it's already a list: `grid[r]`. A **column** is spread across every row, so you collect it:

The **main diagonal** of a square grid runs from top left to bottom right: the cells where the row equals the column. The **other diagonal** runs from top right to bottom left: column `n - 1 - r` in row `r`.

### Transposing

**Transposing** a grid swaps its rows and columns: row 0 becomes column 0. The result has as many rows as the original has columns:

## Neighbours

Many games and simulations need a cell's **neighbours**: the up to 8 cells around it. Loop over the changes in row and column, `dr` and `dc`, each from -1 to 1, skip the cell itself, and skip anything off the edge:

## Practice

## Quiz

## Daily challenge

## Before you go

Draw a 3 by 4 grid on paper and label every cell with its `[r][c]`. Mark a column, a row and the neighbours of an edge cell. Then explain why the Game of Life must build a new grid rather than change the old one in place. Tomorrow: sorting in depth, with key functions, sorting by several things at once, and stability.